# VT-GraF Benchmark: Visible--Thermal Granular Faults under severe clutter
## Comparison: Unsupervised Generalized Frangi Graph vs. Standard Frangi + SAM 2

This notebook reproduces the comparison on the **VT-GraF-Dataset** (multimodal asphalt crack dataset with severe granular clutter) between:
1. **Ours (Generalized Frangi Graph)**: Our unsupervised multimodal graph-based method (using MST + Betweenness Centrality on GPU).
2. **Baseline (Standard Frangi on Thermal + SAM 2 on Visible)**: Zero-shot SAM 2 prompted automatically by standard Frangi filter responses from the thermal modality.

---
### Technical Setup:
- Automatically downloads the **VT-GraF-Dataset** from Google Drive.
- Installs **Segment Anything 2 (SAM 2)** from Meta's repository and downloads the `sam2_hiera_large.pt` weights.
- Imports the custom GPU graph extraction package.
- Runs both methods and outputs comparative metrics: Jaccard (IoU), Tversky index, and Wasserstein distance.


### Connect Google Drive
We mount Google Drive to save and load manual click prompts under `My Drive > Datasets > Raphael`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Environment Setup & Datasets
We install the required libraries (including Meta's SAM 2 repository) and retrieve the pre-trained weights.


In [ ]:
import subprocess
import sys
import os

print("Installing standard packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "POT", "scikit-image", "opencv-python", "matplotlib", "scipy", "torch", "torchvision"])

print("Installing SAM 2...")
# Try standard pip install from Git first
try:
    print("Attempting to install SAM 2 directly from GitHub...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/sam2.git"], check=True, timeout=60)
    print("SAM 2 installed successfully from Git!")
except Exception as e:
    print("Git installation failed. Attempting ZIP download fallback...")
    try:
        # Fallback: download ZIP using wget
        subprocess.run(["wget", "-q", "https://github.com/facebookresearch/sam2/archive/refs/heads/main.zip"], check=True)
        subprocess.run(["unzip", "-q", "main.zip"], check=True)
        # Install from unzipped directory
        subprocess.run([sys.executable, "-m", "pip", "install", "-e", "sam2-main"], check=True)
        print("SAM 2 installed successfully from ZIP fallback!")
        # Clean up zip
        subprocess.run(["rm", "main.zip"])
    except Exception as e2:
        print("\n" + "!"*60)
        print("ERROR: Failed to download and install SAM 2 from GitHub.")
        print("!"*60)
        print("FALLBACK OPTIONS:")
        print("1. Download the ZIP file of SAM 2 on your computer:")
        print("   https://github.com/facebookresearch/sam2/archive/refs/heads/main.zip")
        print("2. Upload the 'main.zip' file to Google Colab files (left sidebar).")
        print("3. Run the following code in a cell:")
        print("   !unzip -q main.zip && pip install -e sam2-main")
        print("!"*60 + "\n")

# Download weights
print("Downloading SAM 2 weights...")
subprocess.run(["wget", "-q", "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt"])
print("Setup complete!")


### Download the VT-GraF-Dataset
If the dataset is not already present, we download it from Google Drive using `gdown`.


In [ ]:
import os
import gdown
from pathlib import Path

folder_id = '1d79CVf9Vqgwwjqn6b2gbc40eu2MM7B7-'
dest_dir = 'VT-GraF-Dataset'

def check_dataset_exists():
    for path in Path('.').rglob('Fissure 1'):
        return True
    return False

if not check_dataset_exists():
    print("Downloading VT-GraF-Dataset from Google Drive...")
    gdown.download_folder(id=folder_id, output=dest_dir, quiet=False, use_cookies=False)
    print("Download completed.")
else:
    print("Dataset already present.")


## 2. Load the Generalized Frangi Graph Modules
We append the package path (handling both local repository execution and Google Colab environments) and load the required modules.


In [ ]:
import sys
import os
import subprocess
from pathlib import Path

def find_isprs_src():
    # 1. Check explicit potential paths first
    paths_to_check = [
        Path('ISPRS'),
        Path('../ISPRS'),
        Path('Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS'),
        Path('/content/drive/MyDrive/Datasets/Raphael/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS'),
        Path('/content/drive/MyDrive/Datasets/Raphael/ISPRS'),
        Path('/content/drive/MyDrive/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS'),
        Path('/content/drive/MyDrive/ISPRS')
    ]
    for p in paths_to_check:
        if p.exists() and (p / 'src').exists():
            return p.resolve()
            
    # 2. Check under /content/drive/MyDrive (if mounted in Colab)
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        try:
            for child in drive_root.iterdir():
                if child.is_dir():
                    # check if child itself is ISPRS
                    if child.name == 'ISPRS' and (child / 'src').exists():
                        return child.resolve()
                    # check if child contains ISPRS/src
                    potential = child / 'ISPRS'
                    if potential.exists() and (potential / 'src').exists():
                        return potential.resolve()
                    # check inside repo name
                    potential2 = child / 'Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset' / 'ISPRS'
                    if potential2.exists() and (potential2 / 'src').exists():
                        return potential2.resolve()
        except Exception:
            pass
            
    # 3. Check under current directory (non-recursive first to avoid hangs)
    for child in Path('.').iterdir():
        if child.is_dir() and 'Generalized-Frangi' in child.name:
            potential = child / 'ISPRS'
            if potential.exists() and (potential / 'src').exists():
                return potential.resolve()
    return None

isprs_path = find_isprs_src()

if isprs_path is None:
    print("Running in Colab: Attempting to clone repository to access code modules...")
    try:
        # Run git clone with a timeout of 30 seconds to avoid hanging forever
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "https://github.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset.git"
        ], check=True, timeout=30)
        
        subprocess.run([
            "git", "-C", "Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset",
            "sparse-checkout", "set", "ISPRS"
        ], check=True)
        
        isprs_path = Path("Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/ISPRS").resolve()
        print("Repository cloned successfully!")
    except Exception as e:
        print("\n" + "!"*60)
        print("ERROR: Failed to clone repository from GitHub (network limit or private repository).")
        print("!"*60)
        print("FALLBACK OPTIONS:")
        print("1. MOUNT GOOGLE DRIVE:")
        print("   If you have the repository on your Google Drive, run the following code in a new cell:")
        print("      from google.colab import drive")
        print("      drive.mount('/content/drive')")
        print("   Then re-run this cell; it will automatically find the files in your Drive.")
        print("\n2. UPLOAD DIRECTLY:")
        print("   Zip the 'ISPRS' folder locally, upload it to the Colab files tab (left sidebar),")
        print("   unzip it in Colab using: !unzip ISPRS.zip")
        print("   Then re-run this cell.")
        print("!"*60 + "\n")
        
if isprs_path is not None:
    sys.path.insert(0, str(isprs_path))
    print(f"Added code modules path: {isprs_path}")
    from src import (
        VTGraFDataset, extract_frangi_graph_gpu, skeletonize_lee,
        thicken, compute_metrics, wasserstein_distance_skeletons
    )
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    dataset = VTGraFDataset('.')
else:
    raise RuntimeError("Could not find or load the 'ISPRS/src' modules. Please follow one of the fallback options above.")


## 3. Initialize SAM 2 Predictor
We load the pre-trained SAM 2 Large model on GPU (or CPU as fallback) and initialize the image predictor.


In [ ]:
import sys
from pathlib import Path

# Try importing from standard paths first, then search for local installation directories
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
except ModuleNotFoundError:
    print("sam2 module not found in default paths. Searching for local installation...")
    potential_paths = [
        Path('sam2-main').resolve(),
        Path('Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/sam2-main').resolve(),
        Path('/content/sam2-main').resolve()
    ]
    # Check Google Drive if mounted
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        try:
            for child in drive_root.iterdir():
                if child.is_dir() and 'sam2' in child.name.lower():
                    potential_paths.append(child.resolve())
        except Exception:
            pass
            
    found = False
    for path in potential_paths:
        if path.exists() and (path / 'sam2').exists():
            sys.path.insert(0, str(path))
            print(f"Added {path} to sys.path")
            try:
                from sam2.build_sam import build_sam2
                from sam2.sam2_image_predictor import SAM2ImagePredictor
                found = True
                print("Import successful after sys.path update!")
                break
            except Exception as e:
                print(f"Import failed with path {path}: {e}")
                
    if not found:
        raise RuntimeError("Could not find or import 'sam2'. Please make sure it is installed and present in your workspace.")

sam2_checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

predictor = SAM2ImagePredictor(build_sam2(model_cfg, sam2_checkpoint, device=device))
print("SAM 2 Large predictor loaded successfully!")


## 4. Define SAM 2 Baseline Prompter
We define the baseline approach which extracts points from standard thermal Frangi responses and queries SAM 2 on the high-resolution visible image.


In [ ]:
import numpy as np
import cv2
from skimage.filters import frangi

def get_manual_points_interactive(image_np, name, num_points=12):
    import sys
    if 'google.colab' in sys.modules:
        import json
        import base64
        import io
        from PIL import Image
        from IPython.display import HTML, display
        import google.colab.output

        img = Image.fromarray(image_np)
        buffered = io.BytesIO()
        img.save(buffered, format="PNG")
        img_b64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
        
        container_id = f"selector_{name.replace(' ', '_')}"
        
        html_code = f"""
        <div id="{container_id}" style="border: 1px solid #ccc; padding: 10px; border-radius: 8px; width: 640px; background-color: #f9f9f9; color: #333; margin-bottom: 20px;">
            <h4 style="margin: 0 0 5px 0;">Interactive Selector: {name}</h4>
            <p style="font-size: 13px; color: #666; margin: 0 0 10px 0;">Click exactly {num_points} points on the fissure path. Click Reset if you make a mistake.</p>
            <div style="position: relative; display: inline-block;">
                <canvas id="{container_id}_canvas" style="cursor: crosshair; display: block;"></canvas>
            </div>
            <div style="margin-top: 10px;">
                <button id="{container_id}_reset" style="padding: 6px 12px; background-color: #f44336; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold;">Reset</button>
                <span id="{container_id}_counter" style="margin-left: 15px; font-weight: bold;">Points: 0 / {num_points}</span>
            </div>
        </div>
        """
        display(HTML(html_code))
        
        js_selector_code = f"""
        new Promise((resolve, reject) => {{
            const img_b64 = "data:image/png;base64,{img_b64}";
            const num_points_target = {num_points};
            const c_id = "{container_id}";
            
            const canvas = document.getElementById(c_id + '_canvas');
            const ctx = canvas.getContext('2d');
            const counter = document.getElementById(c_id + '_counter');
            const btnReset = document.getElementById(c_id + '_reset');
            
            let points = [];
            const img = new Image();
            img.src = img_b64;
            
            img.onload = function() {{
                canvas.width = img.width;
                canvas.height = img.height;
                ctx.drawImage(img, 0, 0);
            }};
            
            function redraw() {{
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                ctx.drawImage(img, 0, 0);
                
                points.forEach((pt, idx) => {{
                    ctx.beginPath();
                    ctx.arc(pt[0], pt[1], 4, 0, 2 * Math.PI);
                    ctx.fillStyle = "red";
                    ctx.fill();
                    ctx.strokeStyle = "white";
                    ctx.lineWidth = 1.5;
                    ctx.stroke();
                    
                    ctx.font = "bold 10px Arial";
                    ctx.fillStyle = "yellow";
                    ctx.fillText(idx + 1, pt[0] + 6, pt[1] - 4);
                }});
                
                counter.innerText = "Points: " + points.length + " / " + num_points_target;
            }}
            
            canvas.addEventListener('click', function(e) {{
                if (points.length >= num_points_target) return;
                
                const rect = canvas.getBoundingClientRect();
                const x = Math.round((e.clientX - rect.left) * (canvas.width / rect.width));
                const y = Math.round((e.clientY - rect.top) * (canvas.height / rect.height));
                
                points.push([x, y]);
                redraw();
                
                if (points.length === num_points_target) {{
                    counter.innerText = "Saved " + num_points_target + " points successfully!";
                    counter.style.color = "green";
                    setTimeout(() => {{
                        resolve(JSON.stringify(points));
                    }}, 500);
                }}
            }});
            
            btnReset.addEventListener('click', function() {{
                points = [];
                redraw();
                counter.style.color = "#333";
            }});
        }})
        """
        pts_json = google.colab.output.eval_js(js_selector_code)
        return json.loads(pts_json)
    else:
        # Local matplotlib ginput fallback
        import matplotlib.pyplot as plt
        print(f"Please click {num_points} points on the pop-up window for {name}...")
        plt.figure(figsize=(10, 8))
        plt.imshow(image_np, cmap='gray')
        plt.title(f"Click exactly {num_points} points for {name}. Press enter when done.")
        plt.axis('on')
        pts = plt.ginput(num_points, timeout=300)
        plt.close()
        return [[int(x), int(y)] for x, y in pts]

def run_baseline_frangi_sam2(sample, predictor, sigmas=[30, 40, 50], num_prompts=12):
    # Retrieve the images
    img_vis_t = sample['visible'] # Torch tensor [0, 1]
    img_ir_t  = sample['infrared'] # Torch tensor [0, 1]
    
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_ir_np = (img_ir_t.numpy() * 255).astype(np.uint8)
    
    # 1. Apply standard Frangi on the thermal image
    # Note: We use sigmas to match our method's scales exactly
    frangi_response = frangi(img_ir_np / 255.0, sigmas=sigmas)
    
    # 2. Threshold the response to get a binary mask
    # We take the top 0.5% intensity pixels to represent the crack
    thresh = np.percentile(frangi_response, 99.5)
    binary_mask = frangi_response > thresh
    
    # 3. Skeletonize to get a thin centerline
    skel = skeletonize_lee(binary_mask)
    
    # 4. Sample point coordinates along the skeleton
    y_coords, x_coords = np.where(skel > 0)
    total_pts = len(y_coords)
    
    if total_pts < 3:
        # Fallback if the Frangi response is empty: sample from max intensity pixel
        y_max, x_max = np.unravel_index(np.argmax(frangi_response), frangi_response.shape)
        pts = np.array([[x_max, y_max]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
    else:
        # Regular spatial sampling
        step = max(1, total_pts // num_prompts)
        pts_x = x_coords[::step][:num_prompts]
        pts_y = y_coords[::step][:num_prompts]
        pts = np.column_stack((pts_x, pts_y)).astype(np.float32)
        labels = np.ones(len(pts), dtype=np.int32) # 1 means positive points
        
    # 5. Predict using SAM 2 on the visible image
    # Convert visible grayscale to RGB (3-channels) for SAM 2
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    predictor.set_image(img_vis_rgb)
    
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return {
        'pred_mask': pred_mask,
        'pts': pts,
        'frangi_response': frangi_response,
        'binary_mask': binary_mask,
        'skel': skel
    }

def run_baseline_manual_sam2(sample, predictor, manual_pts):
    img_vis_t = sample['visible']
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    
    pts = np.array(manual_pts, dtype=np.float32)
    labels = np.ones(len(pts), dtype=np.int32)
    
    predictor.set_image(img_vis_rgb)
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return {
        'pred_mask': pred_mask,
        'pts': pts
    }

def run_baseline_ours_sam2(sample, predictor, ours_skel):
    img_vis_t = sample['visible']
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    
    y_coords, x_coords = np.where(ours_skel > 0)
    total_pts = len(y_coords)
    
    if total_pts == 0:
        # Fallback
        h, w = ours_skel.shape
        pts = np.array([[w // 2, h // 2]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
    else:
        pts = np.column_stack((x_coords, y_coords)).astype(np.float32)
        labels = np.ones(len(pts), dtype=np.int32)
        
    predictor.set_image(img_vis_rgb)
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return {
        'pred_mask': pred_mask,
        'pts': pts
    }


## 5. Quantitative Evaluation & Benchmark
We evaluate both methods on all 5 images of the VT-GraF-Dataset.
- For **Ours (Generalized Frangi Graph)**, we use parameters: `K=2` (dual triangle graph), scale set `Σ=[30,40,50]` and weights `visible: 1/3, infrared: 2/3`. Skeletons are thickened to **5 pixels** for final evaluation.
- For the **Baseline (Frangi Thermal + SAM 2)**, we generate the mask, skeletonize it, and thicken it to **5 pixels** similarly.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
import torch
import numpy as np

def compute_scale_specific_hessian(image_tensor, sigma, device):
    from src.frangi_hessian import FrangiHessianGPU
    fh = FrangiHessianGPU([sigma], device=device)
    ixx, ixy, iyy = fh.compute_hessian(image_tensor, sigma)
    trace = ixx + iyy
    disc = torch.sqrt((ixx - iyy)**2 + 4 * ixy**2)
    spectral_norm_local = (torch.abs(trace) + disc) / 2.0
    max_norm = torch.max(spectral_norm_local) + 1e-8
    
    ixx_n = ixx / max_norm
    iyy_n = iyy / max_norm
    ixy_n = ixy / max_norm
    
    λ1, λ2, θ = fh.compute_eigenvalues_and_vectors(ixx_n, ixy_n, iyy_n)
    S_norm = torch.zeros_like(λ2)
    mask_pos = λ2 > 0
    S_norm[mask_pos] = λ2[mask_pos]
    return S_norm.cpu().numpy()

def compute_fused_scale_specific_hessian(visible_tensor, ir_tensor, weights, sigma, device):
    from src.frangi_hessian import FrangiHessianGPU
    fh = FrangiHessianGPU([sigma], device=device)
    
    fused_ixx = None
    for mod, w in weights.items():
        if w > 0:
            img = visible_tensor if mod == 'visible' else ir_tensor
            ixx, ixy, iyy = fh.compute_hessian(img, sigma)
            trace = ixx + iyy
            disc = torch.sqrt((ixx - iyy)**2 + 4 * ixy**2)
            spectral_norm_local = (torch.abs(trace) + disc) / 2.0
            max_norm = torch.max(spectral_norm_local) + 1e-8
            
            if fused_ixx is None:
                fused_ixx = w * (ixx / max_norm)
                fused_ixy = w * (ixy / max_norm)
                fused_iyy = w * (iyy / max_norm)
            else:
                fused_ixx += w * (ixx / max_norm)
                fused_ixy += w * (ixy / max_norm)
                fused_iyy += w * (iyy / max_norm)
                
    λ1, λ2, θ = fh.compute_eigenvalues_and_vectors(fused_ixx, fused_ixy, fused_iyy)
    S_norm = torch.zeros_like(λ2)
    mask_pos = λ2 > 0
    S_norm[mask_pos] = λ2[mask_pos]
    return S_norm.cpu().numpy()

results = []
visualizations = {}

# Set weights to match the latest notebook: 1/3 Visible, 2/3 Infrared
weights_ours = {'visible': 1/3, 'infrared': 2/3}
sigmes_benchmark = [30, 40, 50]

for i in range(len(dataset)):
    sample = dataset[i]
    name = sample['id']
    print(f"\nProcessing {name}...")
    
    # --- 1. RUN OURS (Generalized Frangi Graph) ---
    imgs_i = {'visible': sample['visible'], 'infrared': sample['infrared']}
    
    # Run the latest code parameters: K=2 and multiscale Σ=[30,40,50]
    max_S_global, sim_img, centrality_i, timings_i, diagnostics_i = extract_frangi_graph_gpu(
        imgs_i, weights_ours, 
        Σ=sigmes_benchmark, R=3, K=2, device=device
    )
    
    pred_ours = (centrality_i > 0.025).astype(np.uint8)
    # Post-process: skeletonize and thicken to 5 pixels (matching latest notebook)
    sk_pred_ours = skeletonize_lee(pred_ours)
    sk_pred_thick_ours = thicken(sk_pred_ours, pixels=5)
    
    # Get Hessian (max over scales of λ2) of each modality individually
    max_S_vis, _, _, _, _ = extract_frangi_graph_gpu(
        imgs_i, {'visible': 1.0, 'infrared': 0.0}, 
        Σ=sigmes_benchmark, R=3, K=2, device=device
    )
    max_S_ir, _, _, _, _ = extract_frangi_graph_gpu(
        imgs_i, {'visible': 0.0, 'infrared': 1.0}, 
        Σ=sigmes_benchmark, R=3, K=2, device=device
    )
    
    # Compute scale-specific Hessians (for 30, 40, 50)
    from skimage.filters import frangi
    scale_visualizations = {}
    for sigma in sigmes_benchmark:
        vis_sig = compute_scale_specific_hessian(imgs_i['visible'], float(sigma), device)
        ir_sig = compute_scale_specific_hessian(imgs_i['infrared'], float(sigma), device)
        fused_sig = compute_fused_scale_specific_hessian(imgs_i['visible'], imgs_i['infrared'], weights_ours, float(sigma), device)
        
        # Classic Frangi response on thermal modality (input already normalized in [0, 1])
        classic_frangi_val = frangi(sample['infrared'].numpy(), sigmas=[float(sigma)])
        
        scale_visualizations[int(sigma)] = {
            'visible': vis_sig,
            'infrared': ir_sig,
            'fused': fused_sig,
            'classic_frangi_ir': classic_frangi_val
        }
    
    # --- 2. RUN BASELINE 1 (Frangi Thermal + SAM 2) ---
    baseline1_outputs = run_baseline_frangi_sam2(sample, predictor, sigmas=sigmes_benchmark, num_prompts=12)
    pred_base1 = baseline1_outputs['pred_mask']
    sk_pred_base1 = skeletonize_lee(pred_base1)
    sk_pred_thick_base1 = thicken(sk_pred_base1, pixels=5)
    
    # --- 3. RUN BASELINE 2 (Manual Prompts + SAM 2) ---
    # Setup path: Google Drive > Datasets > Raphael > Fissure k
    drive_dir = Path("/content/drive/MyDrive/Datasets/Raphael") / name
    os.makedirs(drive_dir, exist_ok=True)
    points_file = drive_dir / "manual_prompts_12.json"
    
    if points_file.exists():
        import json
        with open(points_file, 'r') as f:
            manual_pts = json.load(f)
        print(f"Loaded {len(manual_pts)} manual prompts from Google Drive: {points_file}")
    else:
        # Prompt user manually
        print(f"Manual prompts file not found. Launching interactive selection for {name}...")
        vis_np = (sample['visible'].numpy() * 255).astype(np.uint8)
        manual_pts = get_manual_points_interactive(vis_np, name, num_points=12)
        # Save to Google Drive
        with open(points_file, 'w') as f:
            json.dump(manual_pts, f)
        print(f"Saved selected prompts to Google Drive: {points_file}")
        
    baseline2_outputs = run_baseline_manual_sam2(sample, predictor, manual_pts)
    pred_base2 = baseline2_outputs['pred_mask']
    sk_pred_base2 = skeletonize_lee(pred_base2)
    sk_pred_thick_base2 = thicken(sk_pred_base2, pixels=5)
    
    # --- 4. RUN BASELINE 3 (Ours Prompts + SAM 2) ---
    baseline3_outputs = run_baseline_ours_sam2(sample, predictor, sk_pred_ours)
    pred_base3 = baseline3_outputs['pred_mask']
    sk_pred_base3 = skeletonize_lee(pred_base3)
    sk_pred_thick_base3 = thicken(sk_pred_base3, pixels=5)
    
    # --- 5. GROUND TRUTH SKELETON ---
    gt_arr = sample['gt'].numpy().astype(np.uint8)
    sk_gt = skeletonize_lee(gt_arr)
    sk_gt_thick = thicken(sk_gt, pixels=5)
    
    # --- 6. COMPUTE METRICS ---
    # Ours
    jac_ours, tv_ours = compute_metrics(sk_pred_thick_ours, sk_gt_thick)
    wass_ours = wasserstein_distance_skeletons(sk_pred_thick_ours, sk_gt_thick)
    
    # Baseline 1 (Frangi + SAM 2)
    jac_b1, tv_b1 = compute_metrics(sk_pred_thick_base1, sk_gt_thick)
    wass_b1 = wasserstein_distance_skeletons(sk_pred_thick_base1, sk_gt_thick)
    
    # Baseline 2 (Manual + SAM 2)
    jac_b2, tv_b2 = compute_metrics(sk_pred_thick_base2, sk_gt_thick)
    wass_b2 = wasserstein_distance_skeletons(sk_pred_thick_base2, sk_gt_thick)
    
    # Baseline 3 (Ours Prompts + SAM 2)
    jac_b3, tv_b3 = compute_metrics(sk_pred_thick_base3, sk_gt_thick)
    wass_b3 = wasserstein_distance_skeletons(sk_pred_thick_base3, sk_gt_thick)
    
    results.append({
        'Fissure': name,
        'Ours_Jaccard': jac_ours,
        'Ours_Tversky': tv_ours,
        'Ours_Wasserstein': wass_ours,
        'B1_FrangiSAM2_Jaccard': jac_b1,
        'B1_FrangiSAM2_Tversky': tv_b1,
        'B1_FrangiSAM2_Wasserstein': wass_b1,
        'B2_ManualSAM2_Jaccard': jac_b2,
        'B2_ManualSAM2_Tversky': tv_b2,
        'B2_ManualSAM2_Wasserstein': wass_b2,
        'B3_OursSAM2_Jaccard': jac_b3,
        'B3_OursSAM2_Tversky': tv_b3,
        'B3_OursSAM2_Wasserstein': wass_b3
    })
    
    visualizations[name] = {
        'visible': sample['visible'].numpy(),
        'infrared': sample['infrared'].numpy(),
        'gt': gt_arr,
        'ours_pred': pred_ours,
        'ours': sk_pred_thick_ours,
        'ours_centrality': centrality_i,
        'ours_fused_frangi': max_S_global,
        'ours_vis_frangi': max_S_vis,
        'ours_ir_frangi': max_S_ir,
        'ours_similarity': sim_img,
        'ours_tau_mask': diagnostics_i.get('tau_mask', np.zeros_like(pred_ours)),
        'ours_comp_mask': diagnostics_i.get('comp_mask', np.zeros_like(pred_ours)),
        'baseline1': sk_pred_thick_base1,
        'baseline2': sk_pred_thick_base2,
        'baseline3': sk_pred_thick_base3,
        'points_b1': baseline1_outputs['pts'],
        'points_b2': baseline2_outputs['pts'],
        'points_b3': baseline3_outputs['pts'],
        'baseline_frangi': baseline1_outputs['frangi_response'],
        'scale_visualizations': scale_visualizations
    }

df_res = pd.DataFrame(results)


## 6. Comparison Results Table
We display the quantitative results and compute the mean scores across all fissures for all baselines and our method.


In [ ]:
# Format and display the results
pd.set_option('display.precision', 4)
display(df_res)

# Print Global Averages
print("\n" + "="*50)
print("--- SUMMARY STATISTICS ---")
print("="*50)
print(f"Jaccard (IoU) Mean   : Ours = {df_res['Ours_Jaccard'].mean():.4f} | B1 (Frangi+SAM 2) = {df_res['B1_FrangiSAM2_Jaccard'].mean():.4f} | B2 (Manual+SAM 2) = {df_res['B2_ManualSAM2_Jaccard'].mean():.4f} | B3 (Ours+SAM 2) = {df_res['B3_OursSAM2_Jaccard'].mean():.4f}")
print(f"Tversky Index Mean   : Ours = {df_res['Ours_Tversky'].mean():.4f} | B1 (Frangi+SAM 2) = {df_res['B1_FrangiSAM2_Tversky'].mean():.4f} | B2 (Manual+SAM 2) = {df_res['B2_ManualSAM2_Tversky'].mean():.4f} | B3 (Ours+SAM 2) = {df_res['B3_OursSAM2_Tversky'].mean():.4f}")
print(f"Wasserstein Dist Mean: Ours = {df_res['Ours_Wasserstein'].mean():.4f} | B1 (Frangi+SAM 2) = {df_res['B1_FrangiSAM2_Wasserstein'].mean():.4f} | B2 (Manual+SAM 2) = {df_res['B2_ManualSAM2_Wasserstein'].mean():.4f} | B3 (Ours+SAM 2) = {df_res['B3_OursSAM2_Wasserstein'].mean():.4f}")


## 7. Visual Inspection
We plot each processing step side-by-side to visually inspect the qualitative output.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Create directory to save generated comparison images
os.makedirs('results_images', exist_ok=True)

for name in sorted(visualizations.keys()):
    vis_data = visualizations[name]
    h, w = vis_data['visible'].shape
    
    # --- 1. 4x4 COMPARISON GRID ---
    fig, axes = plt.subplots(4, 4, figsize=(24, 24))
    
    # ROW 1: Raw Inputs, Ground Truth, and Similarity Map
    axes[0, 0].imshow(vis_data['visible'], cmap='gray')
    axes[0, 0].text(15, 35, "Visible Image", color='white', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    axes[0, 1].imshow(vis_data['infrared'], cmap='gray')
    axes[0, 1].text(15, 35, "Inverted Thermal", color='white', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    axes[0, 2].imshow(vis_data['gt'], cmap='gray')
    axes[0, 2].text(15, 35, "Ground Truth", color='white', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    axes[0, 3].imshow(vis_data['ours_similarity'], cmap='magma')
    axes[0, 3].text(15, 35, "Ours: Frangi Similarity Map", color='cyan', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # ROW 2: Hessians (max over scales of λ2 / standard Frangi)
    axes[1, 0].imshow(vis_data['ours_vis_frangi'], cmap='magma')
    axes[1, 0].text(15, 35, "Ours: Vis Hessian (Max)", color='cyan', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    axes[1, 1].imshow(vis_data['ours_ir_frangi'], cmap='magma')
    axes[1, 1].text(15, 35, "Ours: Thermal Hessian (Max)", color='orange', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    axes[1, 2].imshow(vis_data['baseline_frangi'], cmap='magma')
    axes[1, 2].text(15, 35, "Baseline: Classic Frangi (Max)", color='yellow', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    axes[1, 3].imshow(vis_data['ours_fused_frangi'], cmap='magma')
    axes[1, 3].text(15, 35, "Ours: Fused Hessian (Max)", color='magenta', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # ROW 3: Predicted Skeletons & SAM 2 Prompts
    axes[2, 0].imshow(vis_data['ours'], cmap='hot')
    axes[2, 0].text(15, 35, "Ours: Final Prediction Skel", color='cyan', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # Baseline 1 (Frangi + SAM 2)
    axes[2, 1].imshow(vis_data['baseline1'], cmap='hot')
    axes[2, 1].scatter(vis_data['points_b1'][:, 0], vis_data['points_b1'][:, 1], color='cyan', s=45, edgecolors='black')
    axes[2, 1].text(15, 35, "B1 (Frangi+SAM2) Skel & Prompts", color='yellow', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # Baseline 2 (Manual + SAM 2)
    axes[2, 2].imshow(vis_data['baseline2'], cmap='hot')
    axes[2, 2].scatter(vis_data['points_b2'][:, 0], vis_data['points_b2'][:, 1], color='cyan', s=45, edgecolors='black')
    axes[2, 2].text(15, 35, "B2 (Manual+SAM2) Skel & Prompts", color='yellow', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # Baseline 3 (Ours Prompts + SAM 2)
    axes[2, 3].imshow(vis_data['baseline3'], cmap='hot')
    axes[2, 3].scatter(vis_data['points_b3'][:, 0], vis_data['points_b3'][:, 1], color='cyan', s=10, edgecolors='none')
    axes[2, 3].text(15, 35, "B3 (Ours+SAM2) Skel & Prompts", color='yellow', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # ROW 4: Overlays on top of the original visible image
    # Visible + GT
    rgba_gt_only = np.zeros((h, w, 4))
    rgba_gt_only[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.4] # White
    axes[3, 0].imshow(vis_data['visible'], cmap='gray')
    axes[3, 0].imshow(rgba_gt_only)
    axes[3, 0].text(15, 35, "Visible + GT Overlay (White)", color='white', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # Visible + Ours
    rgba_ours_only = np.zeros((h, w, 4))
    rgba_ours_only[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4] # Green
    axes[3, 1].imshow(vis_data['visible'], cmap='gray')
    axes[3, 1].imshow(rgba_ours_only)
    axes[3, 1].text(15, 35, "Visible + Ours Overlay (Green)", color='cyan', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # Visible + B2 Manual SAM 2
    rgba_b2_only = np.zeros((h, w, 4))
    rgba_b2_only[vis_data['baseline2'] > 0] = [1.0, 0.0, 0.0, 0.4] # Red
    axes[3, 2].imshow(vis_data['visible'], cmap='gray')
    axes[3, 2].imshow(rgba_b2_only)
    axes[3, 2].text(15, 35, "Visible + B2 Manual SAM2 (Red)", color='yellow', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # Combined Comparison Overlay
    rgba_combined = np.zeros((h, w, 4))
    rgba_combined[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.4]
    rgba_combined[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4]
    rgba_combined[vis_data['baseline2'] > 0] = [1.0, 0.0, 0.0, 0.4]
    axes[3, 3].imshow(vis_data['visible'], cmap='gray')
    axes[3, 3].imshow(rgba_combined)
    axes[3, 3].text(15, 35, "Overlay (White:GT, Green:Ours, Red:B2)", color='white', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
    
    # Disable axes and titles
    for ax in axes.flat:
        ax.axis('off')
        
    plt.tight_layout()
    
    # Save the clean grid without titles/axes
    grid_filename = f"results_images/comparison_grid_{name.replace(' ', '_')}.png"
    plt.savefig(grid_filename, bbox_inches='tight', pad_inches=0)
    plt.show()
    print(f"Saved comparison grid for {name} to {grid_filename}")
    
    # --- 2. 3x4 SCALE-SPECIFIC HESSIAN GRID ---
    fig_s, axes_s = plt.subplots(3, 4, figsize=(24, 18))
    
    for r_idx, sigma in enumerate([30, 40, 50]):
        scale_data = vis_data['scale_visualizations'][sigma]
        
        # Col 0: Visible scale-specific Hessian
        axes_s[r_idx, 0].imshow(scale_data['visible'], cmap='magma')
        axes_s[r_idx, 0].text(15, 35, f"σ={sigma} | Our Vis Hessian", color='cyan', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
        
        # Col 1: Infrared scale-specific Hessian
        axes_s[r_idx, 1].imshow(scale_data['infrared'], cmap='magma')
        axes_s[r_idx, 1].text(15, 35, f"σ={sigma} | Our Thermal Hessian", color='orange', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
        
        # Col 2: Classic Frangi on Thermal (Infrared)
        axes_s[r_idx, 2].imshow(scale_data['classic_frangi_ir'], cmap='magma')
        axes_s[r_idx, 2].text(15, 35, f"σ={sigma} | Classic Frangi", color='yellow', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
        
        # Col 3: Fused scale-specific Hessian
        axes_s[r_idx, 3].imshow(scale_data['fused'], cmap='magma')
        axes_s[r_idx, 3].text(15, 35, f"σ={sigma} | Our Fused Hessian", color='magenta', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
        
    # Disable axes and titles
    for ax in axes_s.flat:
        ax.axis('off')
        
    plt.tight_layout()
    
    # Save the scale-specific grid without titles/axes
    scale_filename = f"results_images/scales_grid_{name.replace(' ', '_')}.png"
    plt.savefig(scale_filename, bbox_inches='tight', pad_inches=0)
    plt.show()
    print(f"Saved scale-specific grid for {name} to {scale_filename}")
